In [ ]:
import pyodbc

import sqlalchemy

from datetime import datetime

import os
import pandas as pd


server_name ='VIRWSDBS051\V051_01'



conn = pyodbc.connect('Driver={SQL Server};'
                      'Server='+server_name+';'
                      'Database=EDH_PSA_PRD;'
                      'Trusted_Connection=yes;')

file='CUST_FIELD_LIST.txt'

f=open(file,"r")


output_directory = 'C:\\Users\\miw\\Desktop\\py3\\DataProgram\\sqloutput\\'

field_list = f.readlines()

tableName = '[CUSM]'

#print(field_list)

clean_list =[]

cnt = 0

for line in field_list:
    clean_list.append(line.replace('\n', ''))
    
for j in clean_list: 
    
    cnt += 1
    
    print("Job Started at ",datetime.now().strftime("%d/%m/%Y, %H:%M:%S"))
    
    sql_distinct ='select distinct '+j+' FROM [EDH_PSA_PRD].[BANCS].'+tableName+' where [FlexRowIsCurrent]=1'
    print("sql_distinct: ", sql_distinct)
    
    
    
    try:
        df = pd.read_sql(sql_distinct, conn)
    except:
        print("sql failed at ", sql_distinct)
    else:
        csv_output_filename = output_directory+j.replace('[','').replace(']','')+'_distinct.csv'
        df.to_csv(csv_output_filename,sep="|", index=False)
    
    
    sql_groupby ='SELECT '+j+', count(*) as the_count FROM [EDH_PSA_PRD].[BANCS].'+tableName+' where [FlexRowIsCurrent]=1 group by '+j
    print("sql_groupby : ", sql_groupby)
    
    
    try:
        df = pd.read_sql(sql_groupby, conn)
    except:
        print("sql failed at ", sql_groupby)
    else:
        csv_output_filename = output_directory+j.replace('[','').replace(']','')+'_groupby.csv'
        df.to_csv(csv_output_filename,sep="|", index=False)
    

    print("Job Completed for ", j ," at ",datetime.now().strftime("%d/%m/%Y, %H:%M:%S"))
    print("\n")
#     if cnt >  10:
#         break

print("\n","*"*100)
print("ALL query Completed for ", j ," at ",datetime.now().strftime("%d/%m/%Y, %H:%M:%S"))